<a href="https://colab.research.google.com/github/hamzaqarni1/DeepLearning/blob/main/Tutorial_15/Tutorial_15_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from google.colab import files

# --- Set Seed for Reproducibility ---
torch.manual_seed(42)
np.random.seed(42)

print("Starting Tutorial 15 Execution Pipeline...")

# ==============================================================================
# PART 1: GOOGLE STOCK PREDICTION - LSTM vs. SIMPLE RNN (Task 1)
# ==============================================================================
print("\n--- Fetching GOOGL Stock Data ---")
df = yf.download('GOOGL', start='2010-01-01', end='2024-01-01', progress=False)

# Use the 'Close' price and scale it to [0, 1]
data = df['Close'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Split into 80% train, 20% test
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# Create Sequences
def create_sequences(dataset, seq_length):
    X, y = [], []
    for i in range(len(dataset) - seq_length):
        X.append(dataset[i:(i + seq_length)])
        y.append(dataset[i + seq_length])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

sequence_length = 60
X_train, y_train = create_sequences(train_data, sequence_length)
X_test, y_test = create_sequences(test_data, sequence_length)

# --- Define the Models ---
class StockPredictor(nn.Module):
    def __init__(self, rnn_type='LSTM', input_dim=1, hidden_dim=50):
        super(StockPredictor, self).__init__()
        self.rnn_type = rnn_type
        # Using 2 layers to match the tutorial's 2 stacked LSTMs
        if rnn_type == 'LSTM':
            self.recurrent = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        else:
            self.recurrent = nn.RNN(input_dim, hidden_dim, num_layers=2, batch_first=True)

        self.fc1 = nn.Linear(hidden_dim, 25)
        self.fc2 = nn.Linear(25, 1)

    def forward(self, x):
        out, _ = self.recurrent(x)
        # Take the output of the last time step
        out = self.fc1(out[:, -1, :])
        return self.fc2(out)

lstm_model = StockPredictor(rnn_type='LSTM')
rnn_model = StockPredictor(rnn_type='RNN')

def train_stock_model(model, name, epochs=15, lr=0.001):
    print(f"Training {name} Model...")
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_hist = []

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_train)
        loss = criterion(predictions, y_train)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | MSE Loss: {loss.item():.6f}")
    return loss_hist

lstm_losses = train_stock_model(lstm_model, "LSTM")
rnn_losses = train_stock_model(rnn_model, "Simple RNN")

# --- Visual 1: Stock Model Loss Comparison ---
plt.figure(figsize=(9, 5))
plt.plot(lstm_losses, label='LSTM Loss', color='blue', linewidth=2)
plt.plot(rnn_losses, label='Simple RNN Loss', color='orange', linewidth=2)
plt.title("Task 1: LSTM vs. Simple RNN Convergence (GOOGL Stock)", fontweight='bold')
plt.xlabel("Epochs")
plt.ylabel("Mean Squared Error (MSE)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("stock_loss_comparison.png", dpi=300)
plt.close()
files.download("stock_loss_comparison.png")

# --- Visual 2: Prediction vs Actuals ---
lstm_model.eval()
rnn_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(X_test).numpy()
    rnn_preds = rnn_model(X_test).numpy()

# Inverse transform to get actual prices
lstm_preds_scaled = scaler.inverse_transform(lstm_preds)
rnn_preds_scaled = scaler.inverse_transform(rnn_preds)
actual_prices = scaler.inverse_transform(y_test.numpy())

plt.figure(figsize=(12, 6))
plt.plot(actual_prices, label='Actual GOOGL Price', color='black', linewidth=1.5)
plt.plot(lstm_preds_scaled, label='LSTM Prediction', color='blue', alpha=0.8)
plt.plot(rnn_preds_scaled, label='Simple RNN Prediction', color='orange', alpha=0.8)
plt.title("GOOGL Stock Price: Actual vs. Predictions", fontweight='bold')
plt.xlabel("Days (Test Set)")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("stock_predictions_comparison.png", dpi=300)
plt.close()
files.download("stock_predictions_comparison.png")

# Next day prediction
last_sequence = scaled_data[-sequence_length:]
last_sequence_tensor = torch.tensor(last_sequence, dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    next_day_lstm = scaler.inverse_transform(lstm_model(last_sequence_tensor).numpy())
print(f"\nPredicted next day's price for GOOGL (LSTM): ${next_day_lstm[0][0]:.2f}")


# ==============================================================================
# PART 2: SENTIMENT ANALYSIS USING LSTM (Task 2)
# ==============================================================================
print("\n--- Initiating Task 2: LSTM for Sentiment Analysis ---")

# 1. Custom Movie Review Dataset
reviews = [
    "this movie was absolutely wonderful and brilliant",
    "i hated every second of this terrible film",
    "great acting and beautiful visuals loved it",
    "terrible plot and bad acting a waste of time",
    "a masterpiece of modern cinema highly recommended",
    "boring uninspired and completely awful money wasted",
    "what a fantastic and enjoyable experience",
    "do not watch this garbage worst movie ever"
]
# 1 = Positive, 0 = Negative
sentiments = [1, 0, 1, 0, 1, 0, 1, 0]

# Build Vocab
word2idx = {"<PAD>": 0, "<UNK>": 1}
for review in reviews:
    for word in review.split():
        if word not in word2idx:
            word2idx[word] = len(word2idx)

def encode_review(review, max_len=10):
    tokens = [word2idx.get(w, word2idx["<UNK>"]) for w in review.split()]
    if len(tokens) < max_len:
        tokens += [word2idx["<PAD>"]] * (max_len - len(tokens)) # Pad right
    return tokens[:max_len]

X_sent = torch.tensor([encode_review(r) for r in reviews], dtype=torch.long)
y_sent = torch.tensor(sentiments, dtype=torch.float32).unsqueeze(1)

# 2. LSTM Sentiment Architecture
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=50):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        # Grab output of last time step
        final_out = self.fc(out[:, -1, :])
        return final_out

sentiment_model = SentimentLSTM(vocab_size=len(word2idx))
# BCEWithLogitsLoss combines Sigmoid + Binary Cross Entropy safely
criterion_sent = nn.BCEWithLogitsLoss()
optimizer_sent = optim.Adam(sentiment_model.parameters(), lr=0.01)

# 3. Training Loop
sent_loss_hist = []
sentiment_model.train()
for epoch in range(30):
    optimizer_sent.zero_grad()
    logits = sentiment_model(X_sent)
    loss = criterion_sent(logits, y_sent)
    loss.backward()
    optimizer_sent.step()
    sent_loss_hist.append(loss.item())

# --- Visual 3: Sentiment Loss Curve ---
plt.figure(figsize=(7, 4))
plt.plot(sent_loss_hist, color='purple', linewidth=2.5)
plt.title("Task 2: LSTM Sentiment Classifier Convergence", fontweight='bold')
plt.xlabel("Epochs")
plt.ylabel("BCE Loss")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("sentiment_loss_curve.png", dpi=300)
plt.close()
files.download("sentiment_loss_curve.png")

# 4. Testing Unseen Sentences
test_reviews = [
    "loved this beautiful film",
    "terrible and boring waste"
]

sentiment_model.eval()
results = []
with torch.no_grad():
    for r in test_reviews:
        enc = torch.tensor([encode_review(r)], dtype=torch.long)
        logit = sentiment_model(enc)
        prob = torch.sigmoid(logit).item()
        pred_label = "Positive" if prob > 0.5 else "Negative"
        results.append([r, pred_label, f"{prob*100:.1f}%"])

# --- Visual 4: Sentiment Inference Table ---
fig, ax = plt.subplots(figsize=(7, 2))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=results, colLabels=['Unseen Input Text', 'Predicted Class', 'Positive Probability'], loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(weight='bold', color='white')
    elif col == 1 and row > 0:
        cell.set_facecolor('#d4edda' if results[row-1][1] == 'Positive' else '#f8d7da')

plt.title("Task 2: NLP Sentiment LSTM Inference", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("sentiment_results_table.png", dpi=300, bbox_inches='tight')
plt.close()
files.download("sentiment_results_table.png")

print("\nExecution Complete! 4 graphical assets have been downloaded for your LaTeX report.")

Starting Tutorial 15 Execution Pipeline...

--- Fetching GOOGL Stock Data ---


/tmp/ipykernel_588/1457608174.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('GOOGL', start='2010-01-01', end='2024-01-01', progress=False)


Training LSTM Model...
Epoch 5/15 | MSE Loss: 0.022908
Epoch 10/15 | MSE Loss: 0.021762
Epoch 15/15 | MSE Loss: 0.020031
Training Simple RNN Model...
Epoch 5/15 | MSE Loss: 0.022701
Epoch 10/15 | MSE Loss: 0.024088
Epoch 15/15 | MSE Loss: 0.021544


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Execution Complete! 4 graphical assets have been downloaded for your LaTeX report.
